# 14 — Qwen Recipient Profile Memory Training with Unsloth

**Environment:** Google Colab with one NVIDIA CUDA GPU  
**Model:** `unsloth/Qwen3-4B-Base`  
**Purpose:** train a fresh Qwen model to recall explicitly supplied recipient-profile facts for a fixed subset of recipients from the frozen **training split only**.

> **This notebook does not train acute-rejection prediction.**  
> There is no rejection target, no class `0` / class `1` task, and no attempt to predict unseen clinical information.

The aim is deliberately narrower than Notebook 12:

`recipient ID + factual question → Qwen → recorded answer`

This creates a clean factual-memory baseline that can later be evaluated and unlearned separately from the clinical prediction experiment.

## Notebook pipeline

1. Verify the Colab GPU environment.
2. Install and import the required Unsloth, TRL, and metric libraries.
3. Load the Qwen3-4B Base model.
4. Add LoRA so only a small fraction of model parameters are updated.
5. Load the frozen kidney-transplant assessment data and split assignments.
6. Build one recorded profile per recipient from the training split only.
7. Select a deterministic set of 300 memory-training recipients.
8. Reserve 300 separate control recipients that are never used for profile-memory training.
9. Convert the 14 profile facts into direct factual question-answer examples.
10. Train Qwen on all 4,200 memory-recipient questions.
11. Reinforce all 300 memory recipients on six selected factual fields using 1,800 additional questions.
12. Save the trained model, recipient lists, and training contract.
13. Test all memory and unseen-control recipients across all 14 factual fields.
14. Report exact factual recall, F1, precision, recall, balanced accuracy, AUROC, and PR-AUC.
15. Save the detailed results, tables, and fixed F1 threshold for use after unlearning.

## 1. Report the available accelerator

This check reports whether an NVIDIA GPU is available. It does not stop the notebook when no GPU is present.


In [1]:
import shutil
import subprocess
import sys

print("Python:", sys.version.split()[0])

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    print("No NVIDIA GPU detected; this session is running without GPU acceleration.")
else:
    gpu_check = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True,
    )

if nvidia_smi is not None and gpu_check.returncode == 0:
    print("NVIDIA GPU detected.")
    print(gpu_check.stdout)
elif nvidia_smi is not None:
    print("NVIDIA tools were found, but GPU status could not be read.")


Python: 3.13.15
NVIDIA GPU detected.
Tue Sep  8 10:12:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             57W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

## 2. Install the Required Packages

Only the packages needed for this Qwen/Unsloth training notebook are installed. `pandas` is **not** upgraded here because Colab already provides it and upgrading it can create dependency conflicts.

In [2]:
%pip install -q -U unsloth trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

### 2.1 Imports and Reproducibility

A fixed seed is used for recipient selection, dataset shuffling and LoRA initialisation. This means the same 300 memory recipients and 300 control recipients can be recreated later.

In [34]:
from pathlib import Path
import json
import random
import tarfile

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

TRAINING_STACK_AVAILABLE = True
TRAINING_IMPORT_ERROR = None

try:
    from datasets import Dataset
    from unsloth import FastLanguageModel
    from trl import SFTTrainer, SFTConfig
    from transformers import DataCollatorForLanguageModeling
except ModuleNotFoundError as error:
    TRAINING_STACK_AVAILABLE = False
    TRAINING_IMPORT_ERROR = error

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
QWEN_TRAINING_AVAILABLE = CUDA_AVAILABLE and TRAINING_STACK_AVAILABLE

if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable; this session is using CPU only.")

if not TRAINING_STACK_AVAILABLE:
    print(f"Training library unavailable: {TRAINING_IMPORT_ERROR.name}")

if not QWEN_TRAINING_AVAILABLE:
    print("Qwen/Unsloth training cells require both CUDA and the training stack; skip them here.")

CUDA device: NVIDIA A100-SXM4-40GB


## 3. Load Qwen3-4B Base

This notebook starts from the same **base language model family** used in Notebook 12: `unsloth/Qwen3-4B-Base`.

It does **not** load the already kidney-trained classification model. Starting from the fresh Base model keeps this factual-memory experiment separate from acute-rejection prediction.

The maximum sequence length remains `2048`, matching Notebook 12. The profile questions are much shorter than this limit, but keeping the same ceiling avoids changing the model-loading contract unnecessarily.

In [4]:
MODEL_NAME = "unsloth/Qwen3-4B-Base"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", MODEL_NAME)

==((====))==  Unsloth 2026.9.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded model: unsloth/Qwen3-4B-Base


## 4. Add LoRA Fine-Tuning

LoRA adapts Qwen without updating all four billion base-model parameters.

The target modules follow the usual Unsloth language-model LoRA pattern used for the Qwen training workflow: attention projections and MLP projections are adapted, while the pretrained base weights remain frozen.

This is still a language-model fine-tuning task. Qwen keeps its full vocabulary because the required answers include text, identifiers and numbers.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

if hasattr(model, "config"):
    # Transformers models expose this cache setting during training.
    model.config.use_cache = False
else:
    # MLX models do not expose a Transformers-style config object.
    print("MLX model detected; config.use_cache does not apply.")

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
else:
    print("LoRA was applied; this backend does not provide print_trainable_parameters().")

Unsloth 2026.9.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


## 5. Locate the Frozen Project Data

The experiment reuses the permanent dissertation split. It does **not** create a new train/validation/test split.

Only two project artefacts are needed here:

- `kidney_transplant_assessments.csv`
- `split_assignments.csv`

The identity table is not required because this memory task uses the same recorded transplant-profile fields already present in the assessment table. Recipient name is deliberately excluded because the synthetic name contains the same numeric identifier as the recipient ID and could therefore be inferred from the ID pattern rather than genuinely remembered.

In [7]:
REPO_OVERRIDE = None

repo_candidates = [
    Path("/content/qub-machine-unlearning"),
    Path.cwd(),
    Path.cwd().parent,
]

if REPO_OVERRIDE is not None:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (
            path / "code" / "final_submission"
        ).exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Project repository not found. Clone "
        "https://github.com/niamh-hughes/qub-machine-unlearning.git "
        "into /content, then rerun this cell."
    )

FINAL_SUBMISSION_DIR = REPO_ROOT / "code" / "final_submission"
DATA_DIR = FINAL_SUBMISSION_DIR / "data" / "final"
PROCESSED_DIR = FINAL_SUBMISSION_DIR / "processed_data"

ASSESSMENT_PATH = DATA_DIR / "kidney_transplant_assessments.csv"
SPLIT_PATH = PROCESSED_DIR / "split_assignments.csv"

print("Repository:", REPO_ROOT)
print("Assessment data:", ASSESSMENT_PATH)
print("Split assignments:", SPLIT_PATH)

Repository: /content/qub-machine-unlearning
Assessment data: /content/qub-machine-unlearning/code/final_submission/data/final/kidney_transplant_assessments.csv
Split assignments: /content/qub-machine-unlearning/code/final_submission/processed_data/split_assignments.csv


### 5.1 Check the Required Files

The notebook stops immediately if either required input is missing. This avoids accidentally training from a different file or an incomplete checkout.

In [8]:
required_inputs = [
    ASSESSMENT_PATH,
    SPLIT_PATH,
]

input_check = pd.DataFrame({
    "Artefact": [
        "Assessment table",
        "Frozen split assignments",
    ],
    "Path": [str(path) for path in required_inputs],
    "Exists": [path.exists() for path in required_inputs],
})

display(input_check)

missing = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required project files:\n"
        + "\n".join(missing)
    )

,Artefact,Path,Exists
0,Assessment table,/content/qub-machine-unlearning/code/final_sub...,True
1,Frozen split assignments,/content/qub-machine-unlearning/code/final_sub...,True


## 6. Load the Training Split Only

The assessment table is joined to the frozen recipient-level split assignments using `recipient_id` and `donor_id`.

After the merge, only rows labelled `train` are kept for this notebook. Validation and test recipients are not used to create factual-memory examples.

In [9]:
assessments = pd.read_csv(ASSESSMENT_PATH)
split_assignments = pd.read_csv(SPLIT_PATH)

assert len(assessments) == 60_000
assert split_assignments["recipient_id"].is_unique
assert set(split_assignments["split"]) == {
    "train",
    "validation",
    "test",
}

data = assessments.merge(
    split_assignments[
        ["recipient_id", "donor_id", "split"]
    ],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

assert data["split"].notna().all()

train_df = data.loc[
    data["split"].eq("train")
].copy()

print("Training assessments:", len(train_df))
print(
    "Training recipients:",
    train_df["recipient_id"].nunique(),
)

Training assessments: 42024
Training recipients: 7004


## 7. Define the Factual-Memory Fields

The model is trained only on explicit recorded facts. It is **not** asked to infer or predict a future outcome.

The fields are based on the factual profile already used in Notebook 12, with `recipient_name` removed to avoid the synthetic ID/name shortcut.

Fourteen factual fields are used:

- recipient age, sex, ethnicity and region;
- donor ID, age and type;
- kidney failure cause;
- previous-transplant indicator;
- dialysis months;
- ABO compatibility category;
- HLA mismatch count;
- antibody risk score;
- cold ischaemia hours.

In [10]:
QUESTION_SPECS = {
    "recipient_age":
        "What is the recorded recipient age for this recipient?",
    "recipient_sex":
        "What is the recorded recipient sex for this recipient?",
    "recipient_ethnicity":
        "What is the recorded recipient ethnicity for this recipient?",
    "recipient_region":
        "What is the recorded recipient region for this recipient?",
    "donor_id":
        "What is the recorded donor ID for this recipient?",
    "donor_age":
        "What is the recorded donor age for this recipient?",
    "donor_type":
        "What is the recorded donor type for this recipient?",
    "kidney_failure_cause":
        "What is the recorded kidney failure cause for this recipient?",
    "previous_transplant":
        "What is the recorded previous-transplant indicator for this recipient?",
    "dialysis_months":
        "What is the recorded dialysis duration in months for this recipient?",
    "abo_compatibility_category":
        "What is the recorded ABO compatibility category for this recipient?",
    "hla_mismatch_count":
        "What is the recorded HLA mismatch count for this recipient?",
    "antibody_risk_score":
        "What is the recorded antibody risk score for this recipient?",
    "cold_ischaemia_hours":
        "What is the recorded cold ischaemia time in hours for this recipient?",
}

MEMORY_FIELDS = list(QUESTION_SPECS)

missing_fields = [
    field
    for field in MEMORY_FIELDS
    if field not in train_df.columns
]

assert not missing_fields, (
    "Missing factual fields: "
    + ", ".join(missing_fields)
)

assert "acute_rejection_within_30_days" not in MEMORY_FIELDS

print("Factual fields:", len(MEMORY_FIELDS))

Factual fields: 14


## 8. Create One Recorded Profile per Training Recipient

The dataset is longitudinal, so each recipient has repeated assessments.

As in Notebook 12, one deterministic profile row is selected per recipient. The earliest recorded assessment is used. This gives each recipient one fixed set of factual values for the memory task.

In [11]:
recipient_profiles = (
    train_df
    .sort_values(
        ["recipient_id", "assessment_date"]
    )
    .drop_duplicates(
        "recipient_id",
        keep="first",
    )
    .copy()
)

assert recipient_profiles["recipient_id"].is_unique
assert len(recipient_profiles) == 7_004

recipient_profiles = recipient_profiles.set_index(
    "recipient_id"
)

print(
    "Available training profiles:",
    len(recipient_profiles),
)

Available training profiles: 7004


## 9. Select a Fixed Memory Subset

Training on all 7,004 recipient profiles once made the factual task too weak relative to the much larger clinical task in Notebook 12.

This experiment deliberately makes factual memory the **only** training objective and uses a smaller, fixed subset:

- 300 memory-training recipients — their factual questions are shown to Qwen;
- 300 control recipients — selected now, but never shown during this training notebook.

Both groups come from the original frozen training split. Reserving the control group now prevents later evaluation from choosing controls after seeing the results.

In [12]:
MEMORY_RECIPIENT_COUNT = 300
CONTROL_RECIPIENT_COUNT = 300

all_training_recipient_ids = np.array(
    sorted(
        recipient_profiles.index.astype(str)
    )
)

rng = np.random.default_rng(SEED)
permuted_ids = rng.permutation(
    all_training_recipient_ids
)

memory_recipient_ids = permuted_ids[
    :MEMORY_RECIPIENT_COUNT
]

control_recipient_ids = permuted_ids[
    MEMORY_RECIPIENT_COUNT:
    MEMORY_RECIPIENT_COUNT
    + CONTROL_RECIPIENT_COUNT
]

assert len(memory_recipient_ids) == 300
assert len(control_recipient_ids) == 300

assert set(memory_recipient_ids).isdisjoint(
    set(control_recipient_ids)
)

print("Memory-training recipients:", len(memory_recipient_ids))
print("Reserved control recipients:", len(control_recipient_ids))

Memory-training recipients: 300
Reserved control recipients: 300


### 9.1 Verify That the Memory Subset Is Training-Only

This check is important: none of the selected memory recipients may come from the frozen validation or test splits.

In [13]:
memory_split_check = (
    split_assignments
    .loc[
        split_assignments["recipient_id"].isin(
            memory_recipient_ids
        ),
        "split",
    ]
    .value_counts()
)

display(memory_split_check.to_frame("Recipients"))

assert set(memory_split_check.index) == {"train"}
assert int(memory_split_check["train"]) == 300

,Recipients
split,
train,300


### 9.2 Inspect One Selected Profile

This is the information that will later be converted into separate factual questions. The model will **not** receive the full row at query time; each training example contains only the recipient ID, one question and its correct recorded answer.

In [14]:
memory_profiles = (
    recipient_profiles
    .loc[memory_recipient_ids]
    .reset_index()
)

assert not memory_profiles[
    MEMORY_FIELDS
].isna().any().any()

example_columns = [
    "recipient_id",
    *MEMORY_FIELDS,
]

display(
    memory_profiles[
        example_columns
    ].head(1).T
)

,0
recipient_id,V32P-R001887
recipient_age,48
recipient_sex,Male
recipient_ethnicity,Mixed
recipient_region,London
donor_id,V32P-DP001101
donor_age,61
donor_type,Deceased
kidney_failure_cause,Polycystic kidney disease
previous_transplant,0


## 10. Define the Profile-Memory Prompt

The wording deliberately keeps the same simple structure used for factual recall in Notebook 12:

```text
Here is the recipient ID:
<recipient ID>

<question>

SOLUTION
<recorded answer>
```

The important change is that the model now learns **one explicit fact at a time** rather than trying to reproduce a complete 15-line profile in one generation.

At inference time, the prompt stops immediately after `SOLUTION`. The answer must come from what Qwen learned during fine-tuning.

In [15]:
def build_question_prompt(
    recipient_id,
    field,
):
    if field not in QUESTION_SPECS:
        raise KeyError(
            f"Unsupported factual field: {field}"
        )

    question = QUESTION_SPECS[field]

    return f"""Here is the recipient ID:
{recipient_id}

{question}

SOLUTION
"""

### 10.1 Keep Answer Formatting Stable

Integer-valued facts are written without a decimal suffix. Other values are converted to their recorded string form.

Using one deterministic answer representation makes later exact-match evaluation straightforward.

In [16]:
INTEGER_FIELDS = {
    "recipient_age",
    "donor_age",
    "previous_transplant",
    "dialysis_months",
    "hla_mismatch_count",
}

def format_recorded_answer(
    value,
    field,
):
    if field in INTEGER_FIELDS:
        return str(int(value))

    return str(value)

## 11. Create the Factual Question-Answer Training Set

Each of the 300 selected recipients contributes one example for each of the 14 factual fields.

Therefore:

300 recipients × 14 factual questions = 4,200 training examples

Every answer in this dataset is copied directly from the selected recorded profile. No target value is calculated and no clinical outcome is predicted.

In [17]:
qa_rows = []

for _, row in memory_profiles.iterrows():
    recipient_id = row["recipient_id"]

    for field in MEMORY_FIELDS:
        prompt = build_question_prompt(
            recipient_id,
            field,
        )

        answer = format_recorded_answer(
            row[field],
            field,
        )

        qa_rows.append({
            "recipient_id": recipient_id,
            "field": field,
            "question": QUESTION_SPECS[field],
            "answer": answer,
            "text": prompt + answer + tokenizer.eos_token,
        })

qa_examples = pd.DataFrame(qa_rows)

qa_examples = pd.DataFrame(qa_rows)

EXPECTED_QA_EXAMPLES = (
    len(memory_profiles)
    * len(MEMORY_FIELDS)
)

assert len(qa_examples) == EXPECTED_QA_EXAMPLES

assert qa_examples[
    ["recipient_id", "field"]
].duplicated().sum() == 0

print(
    "Factual QA examples:",
    len(qa_examples)
)

Factual QA examples: 4200


### 11.1 Inspect the Exact Training Prompt

This cell prints one complete example exactly as Qwen will see it during supervised fine-tuning.

In [18]:
print(
    qa_examples.iloc[0]["text"]
)

Here is the recipient ID:
V32P-R001887

What is the recorded recipient age for this recipient?

SOLUTION
48<|endoftext|>


### 11.2 Confirm That No Prediction Target Entered the QA Data

The questions should contain only the supported factual profile fields. A rejection-prediction prompt would be a design error in this notebook.

In [19]:
assert not qa_examples[
    "question"
].str.contains(
    "acute rejection",
    case=False,
    regex=False,
).any()

assert set(
    qa_examples["recipient_id"]
).issubset(
    set(memory_recipient_ids)
)

print("Prediction-target check: PASS")
print("Training-only recipient check: PASS")

Prediction-target check: PASS
Training-only recipient check: PASS


## 12. Inspect Sequence Lengths

Before training, the completed QA examples are tokenised once to verify that the `2048`-token model limit is comfortably large enough.

No examples should be truncated.

In [20]:
token_lengths = []

for text in qa_examples["text"]:
    ids = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]

    token_lengths.append(len(ids))

length_summary = pd.Series(
    token_lengths,
    name="token_length",
).describe(
    percentiles=[0.5, 0.9, 0.99]
)

display(length_summary.to_frame())

assert max(token_lengths) <= MAX_SEQ_LENGTH

print(
    "Maximum observed length:",
    max(token_lengths),
)
print("Sequence-length check: PASS")

,token_length
count,4200.000000
mean,36.133571
std,3.200160
min,33.000000
50%,35.000000
90%,42.000000
99%,44.000000
max,44.000000


Maximum observed length: 44
Sequence-length check: PASS


## 13. Train Only on the Recorded Answer

Like Notebook 12, the prompt and instructions are context, not the supervised target.

The custom data collator finds the exact marker:

`SOLUTION\n`

Everything before that marker is masked with `-100`, so it does not contribute to the language-model loss. Only the recorded answer after `SOLUTION` is learned.

Because this notebook contains only one task, the masking logic is simpler than the mixed classification/memory collator in Notebook 12.

In [21]:
class DataCollatorForProfileAnswers(
    DataCollatorForLanguageModeling
):
    """Mask everything before the answer after SOLUTION."""

    def __init__(
        self,
        *args,
        mlm=False,
        ignore_index=-100,
        **kwargs,
    ):
        super().__init__(
            *args,
            mlm=mlm,
            **kwargs,
        )

        self.ignore_index = ignore_index

        self.solution_marker = tokenizer.encode(
            "SOLUTION\n",
            add_special_tokens=False,
        )

    @staticmethod
    def find_subsequence(
        sequence,
        subsequence,
    ):
        match = None

        for start in range(
            len(sequence)
            - len(subsequence)
            + 1
        ):
            if sequence[
                start:
                start + len(subsequence)
            ] == subsequence:
                match = start

        return match

    def torch_call(self, examples):
        batch = super().torch_call(examples)

        for i in range(len(examples)):
            input_ids = batch[
                "input_ids"
            ][i].tolist()

            marker_start = (
                self.find_subsequence(
                    input_ids,
                    self.solution_marker,
                )
            )

            if marker_start is None:
                raise RuntimeError(
                    "SOLUTION marker not found "
                    "in a training example."
                )

            answer_start = (
                marker_start
                + len(self.solution_marker)
            )

            batch[
                "labels"
            ][i, :answer_start] = (
                self.ignore_index
            )

        return batch

### 13.1 Create and Verify the Collator

One example is tokenised and passed through the collator before full training.

The decoded supervised target printed below should contain only the factual answer, not the recipient ID or question.

In [22]:
collator = DataCollatorForProfileAnswers(
    tokenizer=tokenizer,
    mlm=False,
)

sample_encoding = tokenizer(
    qa_examples.iloc[0]["text"],
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
)

sample_batch = collator([
    sample_encoding
])

sample_labels = sample_batch[
    "labels"
][0]

target_token_ids = sample_labels[
    sample_labels.ne(-100)
]

print(
    "Supervised target:",
    tokenizer.decode(
        target_token_ids,
        skip_special_tokens=True,
    ),
)

Supervised target: 48


## 14. Convert the QA Table to an SFT Dataset

The 4,200 examples are shuffled with the fixed seed and converted to a Hugging Face Dataset, which is the input expected by SFTTrainer.

In [23]:
qa_examples = (
    qa_examples
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

train_dataset = Dataset.from_pandas(
    qa_examples[["text"]],
    preserve_index=False,
)

print(train_dataset)

Dataset({
    features: ['text'],
    num_rows: 4200
})


## 15. Configure Qwen Fine-Tuning

The main profile-memory training uses:

- physical batch size `16`;
- gradient accumulation `1`;
- effective batch size `16`;
- learning rate `1e-4`;
- AdamW 8-bit optimiser;
- cosine learning-rate schedule;
- fixed seed `3407`;
- no sequence packing;
- `20` training epochs.

Notebook 12 included factual memory as a small auxiliary task. In this notebook, factual recall is the only training objective.

Each of the 4,200 question-answer examples is presented across 20 epochs. This produces:

`4,200 examples × 20 epochs = 84,000 factual QA exposures`

This deliberately creates a stronger factual-memory baseline for later evaluation and machine unlearning.

In [24]:
TRAIN_EPOCHS = 20
OUTPUT_DIR = "/content/profile_memory_outputs"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    packing=False,

    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        learning_rate=1e-4,

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.00,
        lr_scheduler_type="cosine",

        seed=SEED,
        output_dir=OUTPUT_DIR,
        num_train_epochs=TRAIN_EPOCHS,

        report_to="none",
        group_by_length=True,
    ),

    data_collator=collator,
    dataset_text_field="text",
)

Unsloth: `group_by_length` is not supported by the installed transformers's SFTConfig and will be IGNORED - set `train_sampling_strategy = "group_by_length"` instead.


Unsloth: Tokenizing ["text"]:   0%|          | 0/4200 [00:00<?, ? examples/s]

### 15.1 Training Contract Summary

This cell records the exact design before the expensive training call.

The expected training objective is:

`500 recipients × 14 questions × 5 epochs = 35,000 factual QA exposures`

In [25]:
training_contract = pd.Series({
    "Base model": MODEL_NAME,
    "Memory recipients": len(memory_recipient_ids),
    "Control recipients": len(control_recipient_ids),
    "Questions per recipient": len(MEMORY_FIELDS),
    "Unique QA examples": len(qa_examples),
    "Training epochs": TRAIN_EPOCHS,
    "Total QA exposures":
        len(qa_examples) * TRAIN_EPOCHS,
    "Acute-rejection prediction included": False,
    "Maximum sequence length": MAX_SEQ_LENGTH,
    "Seed": SEED,
})

display(
    training_contract.to_frame("Value")
)

,Value
Base model,unsloth/Qwen3-4B-Base
Memory recipients,300
Control recipients,300
Questions per recipient,14
Unique QA examples,4200
Training epochs,20
Total QA exposures,84000
Acute-rejection prediction included,False
Maximum sequence length,2048
Seed,3407


## 16. Train the Profile-Memory Model

This is the only expensive training step in the notebook.

Qwen receives only the 4,200 factual QA examples created above. The custom collator ensures that loss is calculated only on the recorded answer after `SOLUTION`.

No validation or test records are used.

In [26]:
torch.cuda.empty_cache()

trainer_stats = trainer.train()

print(
    "Training runtime (seconds):",
    trainer_stats.metrics.get(
        "train_runtime"
    ),
)

print(
    "Final training loss:",
    trainer_stats.metrics.get(
        "train_loss"
    ),
)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,200 | Num Epochs = 20 | Total steps = 5,260
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.614437
2,2.724959
3,2.783278
4,3.181988
5,2.431680
6,2.584944
7,2.201286
8,1.911928
9,1.913642
10,1.905341


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_outputs/checkpoint-4000/tokenizer_config.json.
Unsloth: Restored added_t

Training runtime (seconds): 2690.5445
Final training loss: 0.7297404107598751


### 16.1 Inspect the Training Loss

The final rows of the trainer log are shown so the completed run can be reviewed without plotting a large diagnostic section.

In [27]:
loss_rows = [
    {
        "step": item.get("step"),
        "epoch": item.get("epoch"),
        "loss": item.get("loss"),
        "learning_rate": item.get(
            "learning_rate"
        ),
    }
    for item in trainer.state.log_history
    if "loss" in item
]

loss_history = pd.DataFrame(loss_rows)

display(
    loss_history.tail(10)
)

,step,epoch,loss,learning_rate
5250,5251,19.965779,0.410688,8.951995e-10
5251,5252,19.969582,0.527391,7.251120e-10
5252,5253,19.973384,0.518162,5.729283e-10
5253,5254,19.977186,0.603055,4.386484e-10
5254,5255,19.980989,0.533283,3.222725e-10
5255,5256,19.984791,0.549144,2.238004e-10
5256,5257,19.988593,0.774010,1.432323e-10
5257,5258,19.992395,0.889950,8.056818e-11
5258,5259,19.996198,0.569945,3.580808e-11
5259,5260,20.000000,0.682289,8.952022e-12


## 16.2 Focused Factual-Memory Reinforcement

The initial profile-memory training teaches Qwen the factual question format across all 14 profile fields.

To create a stronger and fairer memory baseline for later unlearning, all 300 memory-training recipients receive extra training on six clearly defined recipient-specific facts.

This focused phase continues training the same model rather than restarting from the base model.

The 300 unseen control recipients are not used for training. They are retained only for later evaluation.

In [35]:
FINAL_MEMORY_COUNT = 300

FINAL_MEMORY_FIELDS = [
    "recipient_age",
    "recipient_region",
    "donor_id",
    "donor_age",
    "kidney_failure_cause",
    "dialysis_months",
]

# Reuse all existing memory-training recipients
# for focused reinforcement.
final_memory_recipient_ids = (
    memory_recipient_ids[:FINAL_MEMORY_COUNT]
)

# Keep 300 genuinely unseen controls
# for later evaluation only.
final_control_recipient_ids = (
    control_recipient_ids[:FINAL_MEMORY_COUNT]
)

print(
    "Focused memory recipients:",
    len(final_memory_recipient_ids),
)

print(
    "Focused control recipients:",
    len(final_control_recipient_ids),
)

print(
    "Focused factual fields:",
    len(FINAL_MEMORY_FIELDS),
)

Focused memory recipients: 300
Focused control recipients: 300
Focused factual fields: 6


In [36]:
focused_qa_examples = qa_examples[
    qa_examples["recipient_id"].isin(
        final_memory_recipient_ids
    )
    &
    qa_examples["field"].isin(
        FINAL_MEMORY_FIELDS
    )
].copy()

focused_qa_examples = (
    focused_qa_examples
    .sample(
        frac=1,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

assert len(focused_qa_examples) == 1_800

focused_train_dataset = Dataset.from_pandas(
    focused_qa_examples[["text"]],
    preserve_index=False,
)

print(focused_train_dataset)

Dataset({
    features: ['text'],
    num_rows: 1800
})


In [37]:
FOCUSED_EPOCHS = 20

focused_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=focused_train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    packing=False,

    args=SFTConfig(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,

        learning_rate=1e-4,
        lr_scheduler_type="constant",

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.0,

        seed=SEED,
        output_dir="/content/profile_memory_focused",
        num_train_epochs=FOCUSED_EPOCHS,

        report_to="none",
    ),

    data_collator=collator,
    dataset_text_field="text",
)

focused_stats = focused_trainer.train()

Unsloth: Tokenizing ["text"]:   0%|          | 0/1800 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,800 | Num Epochs = 20 | Total steps = 1,140
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.034630
10,0.698564
15,0.687539
20,0.649165
25,0.603534
30,0.603207
35,0.692071
40,0.528486
45,0.590867
50,0.530934


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_focused/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_focused/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/profile_memory_focused/checkpoint-1140/tokenizer_config.json.


## 17. Save the Trained Profile-Memory Model

The final LoRA-trained model is saved separately from the clinical model produced by Notebook 12.

The model directory is:

`/content/qwen_profile_memory_model`

The model is **not** written into the normal Git repository because model files can be too large for standard Git. Small reproducibility artefacts are saved under the project results directory instead.

In [38]:
PROFILE_MODEL_DIR = Path(
    "/content/qwen_profile_memory_model"
)

PROFILE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(
    PROFILE_MODEL_DIR
)

tokenizer.save_pretrained(
    PROFILE_MODEL_DIR
)

print(
    "Saved model:",
    PROFILE_MODEL_DIR,
)

Saved model: /content/qwen_profile_memory_model


### 17.1 Save the Recipient Subsets and Prompt Contract

The exact 300 memory-training recipient IDs and 300 reserved control-recipient IDs are saved now. Later unlearning runs must reuse these fixed lists rather than resampling recipients.

All 300 memory recipients also receive the focused reinforcement phase on six selected factual fields. The 300 controls are never used for profile-memory training.

The question wording, main-training configuration, and focused-training configuration are saved so the experiment cannot silently change later.

In [39]:
RESULT_DIR = (
    FINAL_SUBMISSION_DIR
    / "results"
    / "qwen_profile_memory"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pd.DataFrame({
    "recipient_id": memory_recipient_ids
}).to_csv(
    RESULT_DIR
    / "memory_training_recipient_ids.csv",
    index=False,
)

pd.DataFrame({
    "recipient_id": control_recipient_ids
}).to_csv(
    RESULT_DIR
    / "memory_control_recipient_ids.csv",
    index=False,
)

contract = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "memory_recipient_count":
        MEMORY_RECIPIENT_COUNT,
    "control_recipient_count":
        CONTROL_RECIPIENT_COUNT,
    "memory_fields": MEMORY_FIELDS,
    "question_specs": QUESTION_SPECS,
    "training_epochs": TRAIN_EPOCHS,
    "unique_qa_examples": len(qa_examples),
    "focused_memory_recipient_count":
        len(final_memory_recipient_ids),
    "focused_control_recipient_count":
        len(final_control_recipient_ids),
    "focused_memory_fields":
        FINAL_MEMORY_FIELDS,
    "focused_unique_qa_examples":
        len(focused_qa_examples),
    "focused_training_epochs":
        FOCUSED_EPOCHS,
    "prediction_task_included": False,
    "answer_marker": "SOLUTION\n",
}

with open(
    RESULT_DIR / "profile_memory_contract.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        contract,
        handle,
        indent=2,
    )

print("Saved reproducibility files to:")
print(RESULT_DIR)

Saved reproducibility files to:
/content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory


### 17.2 Package the Model for Download

A `.tar.gz` archive is created in `/content` so the completed model can be downloaded from Colab and restored later.

This archive should be kept separate from the Notebook 12 clinical model.

In [40]:
MODEL_ARCHIVE = Path(
    "/content/qwen_profile_memory_model.tar.gz"
)

with tarfile.open(
    MODEL_ARCHIVE,
    "w:gz",
) as archive:
    archive.add(
        PROFILE_MODEL_DIR,
        arcname=PROFILE_MODEL_DIR.name,
    )

print("Created archive:")
print(MODEL_ARCHIVE)
print(
    "Archive size (MB):",
    round(
        MODEL_ARCHIVE.stat().st_size
        / (1024 ** 2),
        2,
    ),
)

Created archive:
/content/qwen_profile_memory_model.tar.gz
Archive size (MB): 235.38


## 18. Quick Manual Memory Test

This section provides a simple way to test whether the trained Qwen model can recall one of the factual values it was explicitly shown during training.

The test works as follows:

1. Choose one recipient from the 300 memory-training recipients.
2. Choose one factual field.
3. Give Qwen only the recipient ID and the question.
4. Let Qwen generate its answer.
5. Look up the recorded answer afterwards and compare the two.

The correct answer is **not included in the prompt given to Qwen**.

In [41]:
test_options = pd.DataFrame({
    "field_key": MEMORY_FIELDS,
    "question": [
        QUESTION_SPECS[field]
        for field in MEMORY_FIELDS
    ],
})

display(test_options)

print("\nExample trained recipient IDs:")

for recipient_id in memory_recipient_ids[:10]:
    print(" -", recipient_id)

,field_key,question
0,recipient_age,What is the recorded recipient age for this re...
1,recipient_sex,What is the recorded recipient sex for this re...
2,recipient_ethnicity,What is the recorded recipient ethnicity for t...
3,recipient_region,What is the recorded recipient region for this...
4,donor_id,What is the recorded donor ID for this recipient?
5,donor_age,What is the recorded donor age for this recipi...
6,donor_type,What is the recorded donor type for this recip...
7,kidney_failure_cause,What is the recorded kidney failure cause for ...
8,previous_transplant,What is the recorded previous-transplant indic...
9,dialysis_months,What is the recorded dialysis duration in mont...



Example trained recipient IDs:
 - V32P-R001887
 - V32P-R001786
 - V32P-R000158
 - V32P-R004271
 - V32P-R007479
 - V32P-R002063
 - V32P-R005930
 - V32P-R002276
 - V32P-R003206
 - V32P-R007238


### 18.1 Function Used to Ask Qwen

The function below sends a factual question to the trained model.

It only accepts:

- a recipient from either the 300 memory-training recipients or the 300 fixed unseen controls;
- one of the 14 factual fields used during training.

The prompt contains the recipient ID and question, but not the correct answer.

The unseen controls are permitted here only for evaluation. They were never included in either profile-memory training phase.

In [42]:
MEMORY_RECIPIENT_SET = set(
    memory_recipient_ids
)

CONTROL_RECIPIENT_SET = set(
    final_control_recipient_ids
)

EVALUATION_RECIPIENT_SET = (
    MEMORY_RECIPIENT_SET
    | CONTROL_RECIPIENT_SET
)


def ask_profile_fact(
    recipient_id,
    field,
):
    # Only allow recipients in one of the
    # fixed memory-evaluation groups.
    if recipient_id not in EVALUATION_RECIPIENT_SET:
        raise ValueError(
            "This recipient is not in the fixed "
            "memory or unseen-control evaluation groups."
        )

    # Only allow factual question types
    # that were used during training.
    if field not in QUESTION_SPECS:
        raise ValueError(
            "Unsupported field. Choose one of: "
            + ", ".join(MEMORY_FIELDS)
        )

    # Build the same prompt format used during training.
    prompt = build_question_prompt(
        recipient_id,
        field,
    )

    # Put Qwen into inference mode.
    FastLanguageModel.for_inference(model)

    # Remove any stored max_new_tokens setting.
    model.generation_config.max_new_tokens = None

    # Tokenise the question.
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    # Give Qwen room for 12 answer tokens.
    generation_limit = (
        encoded["input_ids"].shape[1] + 12
    )

    # Generate Qwen's answer.
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_length=generation_limit,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Keep only the tokens generated by Qwen,
    # not the original prompt.
    new_tokens = generated[
        0,
        encoded["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    # All expected factual answers are one line.
    return answer.splitlines()[0].strip()

### 18.2 Test the Model Here

This is the only cell that needs to be edited when manually testing the model.

Change:

- `TEST_RECIPIENT_ID` to one of the trained recipient IDs shown above;
- `TEST_FIELD` to one of the supported field keys.

Qwen generates its answer first. The real recorded answer is retrieved afterwards only for comparison.

In [43]:
# =========================================================
# EDIT ONLY THESE TWO VALUES
# =========================================================

TEST_RECIPIENT_ID = memory_recipient_ids[0]
TEST_FIELD = "dialysis_months"

# =========================================================
# NO EDITING NEEDED BELOW
# =========================================================

question = QUESTION_SPECS[
    TEST_FIELD
]

# ---------------------------------------------------------
# 1. Ask Qwen
# ---------------------------------------------------------

model_answer = ask_profile_fact(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
)

# ---------------------------------------------------------
# 2. Get the actual recorded answer AFTER generation
# ---------------------------------------------------------

expected_value = recipient_profiles.loc[
    TEST_RECIPIENT_ID,
    TEST_FIELD,
]

expected_answer = format_recorded_answer(
    expected_value,
    TEST_FIELD,
)

# ---------------------------------------------------------
# 3. Compare the answers
# ---------------------------------------------------------

exact_match = (
    model_answer.strip().casefold()
    == expected_answer.strip().casefold()
)

result = pd.DataFrame([{
    "Recipient ID": TEST_RECIPIENT_ID,
    "Field": TEST_FIELD,
    "Question": question,
    "Qwen answer": model_answer,
    "Recorded answer": expected_answer,
    "Exact match": exact_match,
}])

display(result)

print(
    "\nResult:",
    "CORRECT" if exact_match else "INCORRECT",
)

,Recipient ID,Field,Question,Qwen answer,Recorded answer,Exact match
0,V32P-R001887,dialysis_months,What is the recorded dialysis duration in mont...,10,10,True



Result: CORRECT


In [44]:
def score_answer(recipient_id, field, candidate_answer):
    prompt = build_question_prompt(
        recipient_id,
        field,
    )

    # Tokenise prompt and candidate answer.
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    answer_ids = tokenizer(
        str(candidate_answer),
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    # Combine them.
    input_ids = torch.cat(
        [prompt_ids, answer_ids],
        dim=1,
    )

    FastLanguageModel.for_inference(model)

    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids
        )

    logits = outputs.logits

    # Get probabilities for the answer tokens only.
    answer_start = prompt_ids.shape[1]

    token_log_probs = []

    for i in range(answer_ids.shape[1]):
        position = answer_start + i - 1

        probs = torch.log_softmax(
            logits[0, position],
            dim=-1,
        )

        token_id = answer_ids[0, i]

        token_log_probs.append(
            probs[token_id].item()
        )

    return np.mean(token_log_probs)

In [45]:
TEST_RECIPIENT_ID = "V32P-R001887"
TEST_FIELD = "donor_age"

# Real recorded answer
correct_value = recipient_profiles.loc[
    TEST_RECIPIENT_ID,
    TEST_FIELD,
]

correct_answer = format_recorded_answer(
    correct_value,
    TEST_FIELD,
)

# What Qwen generates
generated_answer = ask_profile_fact(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
)

correct_score = score_answer(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
    correct_answer,
)

generated_score = score_answer(
    TEST_RECIPIENT_ID,
    TEST_FIELD,
    generated_answer,
)

print("Correct answer:", correct_answer)
print("Qwen generated:", generated_answer)

print("\nCorrect-answer score:", correct_score)
print("Generated-answer score:", generated_score)

Correct answer: 61
Qwen generated: 61

Correct-answer score: -0.0006775975853088312
Generated-answer score: -0.0006775975853088312


## 18.3 Full Memory and Unseen-Control Evaluation

This section measures factual recall across the fixed evaluation groups.

- **Memory group:** 300 recipients used in profile-memory training and focused reinforcement.
- **Unseen-control group:** 300 recipients never used in either profile-memory training phase.
- **Questions:** all 14 factual fields are asked for every recipient.
- **Total evaluation questions:** 8,400.

For every question, the notebook records:

1. the real recorded answer;
2. Qwen's generated answer;
3. whether the answer is an exact match;
4. Qwen's confidence in the real answer.

The results are used to report factual-recall accuracy and membership metrics: F1, precision, recall, balanced accuracy, AUROC, and PR-AUC.

These metrics measure profile-memory evidence, not acute-rejection prediction.

In [46]:
CALIBRATION_RECIPIENTS_PER_GROUP = 60

rng = np.random.default_rng(SEED)

calibration_memory_ids = set(
    rng.choice(
        memory_recipient_ids,
        size=CALIBRATION_RECIPIENTS_PER_GROUP,
        replace=False,
    ).tolist()
)

calibration_control_ids = set(
    rng.choice(
        final_control_recipient_ids,
        size=CALIBRATION_RECIPIENTS_PER_GROUP,
        replace=False,
    ).tolist()
)

evaluation_rows = []

for group, recipient_ids, membership_label, calibration_ids in [
    (
        "memory",
        memory_recipient_ids,
        1,
        calibration_memory_ids,
    ),
    (
        "unseen_control",
        final_control_recipient_ids,
        0,
        calibration_control_ids,
    ),
]:
    for recipient_id in recipient_ids:
        for field in MEMORY_FIELDS:
            recorded_value = recipient_profiles.loc[
                recipient_id,
                field,
            ]

            evaluation_rows.append({
                "recipient_id": recipient_id,
                "group": group,
                "membership_label": membership_label,
                "evaluation_split": (
                    "calibration"
                    if recipient_id in calibration_ids
                    else "test"
                ),
                "field": field,
                "question": QUESTION_SPECS[field],
                "recorded_answer": format_recorded_answer(
                    recorded_value,
                    field,
                ),
                "focused_field": (
                    field in FINAL_MEMORY_FIELDS
                ),
            })

evaluation_plan = pd.DataFrame(evaluation_rows)

assert len(evaluation_plan) == 8_400
assert (
    evaluation_plan["group"].value_counts().to_dict()
    == {
        "memory": 4_200,
        "unseen_control": 4_200,
    }
)

EVALUATION_PLAN_PATH = (
    RESULT_DIR
    / "profile_memory_evaluation_plan.csv"
)

evaluation_plan.to_csv(
    EVALUATION_PLAN_PATH,
    index=False,
)

display(evaluation_plan.head())

print(
    "Memory questions:",
    (evaluation_plan["group"] == "memory").sum(),
)

print(
    "Unseen-control questions:",
    (
        evaluation_plan["group"]
        == "unseen_control"
    ).sum(),
)

print(
    "Calibration recipients per group:",
    CALIBRATION_RECIPIENTS_PER_GROUP,
)

print(
    "Test recipients per group:",
    len(memory_recipient_ids)
    - CALIBRATION_RECIPIENTS_PER_GROUP,
)

,recipient_id,group,membership_label,evaluation_split,field,question,recorded_answer,focused_field
0,V32P-R001887,memory,1,test,recipient_age,What is the recorded recipient age for this re...,48,True
1,V32P-R001887,memory,1,test,recipient_sex,What is the recorded recipient sex for this re...,Male,False
2,V32P-R001887,memory,1,test,recipient_ethnicity,What is the recorded recipient ethnicity for t...,Mixed,False
3,V32P-R001887,memory,1,test,recipient_region,What is the recorded recipient region for this...,London,True
4,V32P-R001887,memory,1,test,donor_id,What is the recorded donor ID for this recipient?,V32P-DP001101,True


Memory questions: 4200
Unseen-control questions: 4200
Calibration recipients per group: 60
Test recipients per group: 240


### 18.4 Generate and Score Every Answer

Qwen is now asked every question in the fixed evaluation plan.

For each question, the notebook saves:

- Qwen's generated answer;
- whether it exactly matches the recorded answer;
- the mean log-probability assigned to the recorded answer.

The mean log-probability is the continuous confidence score used for AUROC and PR-AUC: a higher value means Qwen assigned more confidence to the correct answer.

Progress is saved every 25 questions. If the runtime stops, rerunning the next cell resumes from the saved results file rather than starting again.

In [47]:
EVALUATION_RUN_NAME = "all_300_reinforced_v1"

EVALUATION_RESULTS_PATH = (
    RESULT_DIR
    / (
        "profile_memory_evaluation_rows_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

RESULT_KEY_COLUMNS = [
    "recipient_id",
    "field",
]


def normalise_answer(answer):
    return " ".join(
        str(answer).strip().casefold().split()
    )


assert not evaluation_plan.duplicated(
    RESULT_KEY_COLUMNS
).any()

if EVALUATION_RESULTS_PATH.exists():
    existing_results = pd.read_csv(
        EVALUATION_RESULTS_PATH
    )

    missing_columns = set(
        RESULT_KEY_COLUMNS
    ) - set(existing_results.columns)

    if missing_columns:
        raise ValueError(
            "Existing evaluation file has the wrong "
            "format. Change EVALUATION_RUN_NAME before "
            "running this cell again."
        )

    existing_results = (
        existing_results
        .drop_duplicates(
            RESULT_KEY_COLUMNS,
            keep="last",
        )
        .copy()
    )
else:
    existing_results = pd.DataFrame()

completed_keys = set()

if not existing_results.empty:
    completed_keys = set(
        existing_results[
            RESULT_KEY_COLUMNS
        ].itertuples(
            index=False,
            name=None,
        )
    )

new_result_rows = []

print(
    "Previously completed questions:",
    len(completed_keys),
)

for _, plan_row in evaluation_plan.iterrows():
    result_key = (
        plan_row["recipient_id"],
        plan_row["field"],
    )

    if result_key in completed_keys:
        continue

    generated_answer = ask_profile_fact(
        plan_row["recipient_id"],
        plan_row["field"],
    )

    correct_answer_score = score_answer(
        plan_row["recipient_id"],
        plan_row["field"],
        plan_row["recorded_answer"],
    )

    exact_match = (
        normalise_answer(generated_answer)
        == normalise_answer(
            plan_row["recorded_answer"]
        )
    )

    result_row = plan_row.to_dict()

    result_row.update({
        "generated_answer": generated_answer,
        "exact_match": exact_match,
        "correct_answer_log_probability":
            correct_answer_score,
    })

    new_result_rows.append(result_row)

    if len(new_result_rows) % 25 == 0:
        evaluation_results = pd.concat(
            [
                existing_results,
                pd.DataFrame(new_result_rows),
            ],
            ignore_index=True,
        )

        evaluation_results.to_csv(
            EVALUATION_RESULTS_PATH,
            index=False,
        )

        print(
            "Saved:",
            len(evaluation_results),
            "/",
            len(evaluation_plan),
            "questions",
        )

evaluation_results = pd.concat(
    [
        existing_results,
        pd.DataFrame(new_result_rows),
    ],
    ignore_index=True,
)

evaluation_results = (
    evaluation_results
    .drop_duplicates(
        RESULT_KEY_COLUMNS,
        keep="last",
    )
    .sort_values(
        ["group", "recipient_id", "field"]
    )
    .reset_index(drop=True)
)

evaluation_results.to_csv(
    EVALUATION_RESULTS_PATH,
    index=False,
)

assert len(evaluation_results) == len(
    evaluation_plan
)

print(
    "Evaluation complete:",
    len(evaluation_results),
    "questions",
)

print("Saved results to:")
print(EVALUATION_RESULTS_PATH)

Previously completed questions: 0
Saved: 25 / 8400 questions
Saved: 50 / 8400 questions
Saved: 75 / 8400 questions
Saved: 100 / 8400 questions
Saved: 125 / 8400 questions
Saved: 150 / 8400 questions
Saved: 175 / 8400 questions
Saved: 200 / 8400 questions
Saved: 225 / 8400 questions
Saved: 250 / 8400 questions
Saved: 275 / 8400 questions
Saved: 300 / 8400 questions
Saved: 325 / 8400 questions
Saved: 350 / 8400 questions
Saved: 375 / 8400 questions
Saved: 400 / 8400 questions
Saved: 425 / 8400 questions
Saved: 450 / 8400 questions
Saved: 475 / 8400 questions
Saved: 500 / 8400 questions
Saved: 525 / 8400 questions
Saved: 550 / 8400 questions
Saved: 575 / 8400 questions
Saved: 600 / 8400 questions
Saved: 625 / 8400 questions
Saved: 650 / 8400 questions
Saved: 675 / 8400 questions
Saved: 700 / 8400 questions
Saved: 725 / 8400 questions
Saved: 750 / 8400 questions
Saved: 775 / 8400 questions
Saved: 800 / 8400 questions
Saved: 825 / 8400 questions
Saved: 850 / 8400 questions
Saved: 875 / 8400

### 18.5 Results Tables and Membership Metrics

The primary result is exact factual-recall accuracy: how often Qwen generated the recorded profile answer.

The membership metrics use the correct-answer confidence score:

- a score is calculated for every question;
- 60 memory recipients and 60 unseen controls are used only to choose one fixed F1 threshold;
- F1, precision, recall, balanced accuracy, AUROC, and PR-AUC are then reported on the remaining 240 memory and 240 unseen-control recipients;
- the selected threshold is saved and must be reused unchanged after unlearning.

This prevents the F1 result from being inflated by choosing its threshold on the same test questions being reported.

In [49]:
OVERALL_RECALL_PATH = (
    RESULT_DIR
    / (
        "profile_memory_overall_recall_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

FIELD_RECALL_PATH = (
    RESULT_DIR
    / (
        "profile_memory_recall_by_field_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

FOCUSED_FIELD_RECALL_PATH = (
    RESULT_DIR
    / (
        "profile_memory_recall_by_field_type_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

overall_recall_table = (
    evaluation_results
    .groupby(
        "group",
        as_index=False,
    )
    .agg(
        recipients=(
            "recipient_id",
            "nunique",
        ),
        questions=(
            "exact_match",
            "size",
        ),
        correct_answers=(
            "exact_match",
            "sum",
        ),
        exact_match_accuracy=(
            "exact_match",
            "mean",
        ),
    )
)

overall_recall_table[
    "exact_match_accuracy_pct"
] = (
    100
    * overall_recall_table[
        "exact_match_accuracy"
    ]
).round(2)

memory_accuracy = overall_recall_table.loc[
    overall_recall_table["group"] == "memory",
    "exact_match_accuracy_pct",
].iloc[0]

control_accuracy = overall_recall_table.loc[
    overall_recall_table["group"]
    == "unseen_control",
    "exact_match_accuracy_pct",
].iloc[0]

overall_comparison = pd.DataFrame([{
    "memory_exact_match_accuracy_pct":
        memory_accuracy,
    "unseen_control_exact_match_accuracy_pct":
        control_accuracy,
    "memory_minus_control_percentage_points":
        round(
            memory_accuracy - control_accuracy,
            2,
        ),
}])

field_recall_long = (
    evaluation_results
    .groupby(
        ["field", "group"],
        as_index=False,
    )
    .agg(
        questions=(
            "exact_match",
            "size",
        ),
        correct_answers=(
            "exact_match",
            "sum",
        ),
        exact_match_accuracy=(
            "exact_match",
            "mean",
        ),
    )
)

field_recall_long[
    "exact_match_accuracy_pct"
] = (
    100
    * field_recall_long[
        "exact_match_accuracy"
    ]
).round(2)

field_recall_table = (
    field_recall_long
    .pivot(
        index="field",
        columns="group",
        values="exact_match_accuracy_pct",
    )
    .reindex(MEMORY_FIELDS)
    .reset_index()
)

field_recall_table.columns.name = None

field_recall_table[
    "memory_minus_unseen_control_percentage_points"
] = (
    field_recall_table["memory"]
    - field_recall_table["unseen_control"]
).round(2)

focused_field_recall_table = (
    evaluation_results
    .groupby(
        ["focused_field", "group"],
        as_index=False,
    )
    .agg(
        questions=(
            "exact_match",
            "size",
        ),
        correct_answers=(
            "exact_match",
            "sum",
        ),
        exact_match_accuracy=(
            "exact_match",
            "mean",
        ),
    )
)

focused_field_recall_table[
    "exact_match_accuracy_pct"
] = (
    100
    * focused_field_recall_table[
        "exact_match_accuracy"
    ]
).round(2)

overall_recall_table.to_csv(
    OVERALL_RECALL_PATH,
    index=False,
)

field_recall_table.to_csv(
    FIELD_RECALL_PATH,
    index=False,
)

focused_field_recall_table.to_csv(
    FOCUSED_FIELD_RECALL_PATH,
    index=False,
)

print("Overall factual-recall results:")
display(overall_recall_table)

print("Memory versus unseen-control difference:")
display(overall_comparison)

print("Recall by factual field:")
display(field_recall_table)

print("Recall for focused versus other fields:")
display(focused_field_recall_table)

Overall factual-recall results:


,group,recipients,questions,correct_answers,exact_match_accuracy,exact_match_accuracy_pct
0,memory,300,4200,2660,0.633333,63.33
1,unseen_control,300,4200,1015,0.241667,24.17


Memory versus unseen-control difference:


,memory_exact_match_accuracy_pct,unseen_control_exact_match_accuracy_pct,memory_minus_control_percentage_points
0,63.33,24.17,39.16


Recall by factual field:


,field,memory,unseen_control,memory_minus_unseen_control_percentage_points
0,recipient_age,99.00,3.00,96.00
1,recipient_sex,54.33,51.67,2.66
2,recipient_ethnicity,19.00,20.00,-1.00
3,recipient_region,100.00,21.67,78.33
4,donor_id,94.00,0.00,94.00
5,donor_age,98.67,2.33,96.34
6,donor_type,59.67,59.00,0.67
7,kidney_failure_cause,100.00,21.00,79.00
8,previous_transplant,89.33,87.33,2.00
9,dialysis_months,100.00,1.67,98.33


Recall for focused versus other fields:


,focused_field,group,questions,correct_answers,exact_match_accuracy,exact_match_accuracy_pct
0,False,memory,2400,885,0.368750,36.88
1,False,unseen_control,2400,866,0.360833,36.08
2,True,memory,1800,1775,0.986111,98.61
3,True,unseen_control,1800,149,0.082778,8.28


In [50]:
MEMBERSHIP_METRICS_PATH = (
    RESULT_DIR
    / (
        "profile_memory_membership_metrics_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

MEMBERSHIP_THRESHOLD_PATH = (
    RESULT_DIR
    / (
        "profile_memory_membership_threshold_"
        + EVALUATION_RUN_NAME
        + ".json"
    )
)

MEMBERSHIP_TEST_ROWS_PATH = (
    RESULT_DIR
    / (
        "profile_memory_membership_test_rows_"
        + EVALUATION_RUN_NAME
        + ".csv"
    )
)

calibration_results = (
    evaluation_results[
        evaluation_results["evaluation_split"]
        == "calibration"
    ]
    .copy()
)

test_results = (
    evaluation_results[
        evaluation_results["evaluation_split"]
        == "test"
    ]
    .copy()
)

for result_frame in [
    calibration_results,
    test_results,
]:
    result_frame[
        "correct_answer_log_probability"
    ] = pd.to_numeric(
        result_frame[
            "correct_answer_log_probability"
        ],
        errors="raise",
    )

calibration_scores = calibration_results[
    "correct_answer_log_probability"
].to_numpy()

calibration_labels = calibration_results[
    "membership_label"
].to_numpy()

threshold_rows = []

for threshold in np.unique(calibration_scores):
    calibration_predictions = (
        calibration_scores >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": float(threshold),
        "f1": f1_score(
            calibration_labels,
            calibration_predictions,
            zero_division=0,
        ),
    })

threshold_search = pd.DataFrame(
    threshold_rows
)

best_threshold_row = (
    threshold_search
    .sort_values(
        ["f1", "threshold"],
        ascending=[False, False],
    )
    .iloc[0]
)

membership_threshold = float(
    best_threshold_row["threshold"]
)

test_scores = test_results[
    "correct_answer_log_probability"
].to_numpy()

test_labels = test_results[
    "membership_label"
].to_numpy()

test_predictions = (
    test_scores >= membership_threshold
).astype(int)

membership_metrics = pd.DataFrame([{
    "evaluation_run_name":
        EVALUATION_RUN_NAME,
    "score_definition":
        "mean correct-answer log probability",
    "f1_threshold":
        membership_threshold,
    "calibration_recipients_per_group":
        CALIBRATION_RECIPIENTS_PER_GROUP,
    "test_recipients_per_group":
        len(memory_recipient_ids)
        - CALIBRATION_RECIPIENTS_PER_GROUP,
    "test_questions":
        len(test_results),
    "f1":
        f1_score(
            test_labels,
            test_predictions,
            zero_division=0,
        ),
    "precision":
        precision_score(
            test_labels,
            test_predictions,
            zero_division=0,
        ),
    "recall":
        recall_score(
            test_labels,
            test_predictions,
            zero_division=0,
        ),
    "balanced_accuracy":
        balanced_accuracy_score(
            test_labels,
            test_predictions,
        ),
    "auroc":
        roc_auc_score(
            test_labels,
            test_scores,
        ),
    "pr_auc":
        average_precision_score(
            test_labels,
            test_scores,
        ),
}])

test_membership_rows = test_results.copy()

test_membership_rows[
    "predicted_membership"
] = test_predictions

threshold_search.to_csv(
    RESULT_DIR
    / (
        "profile_memory_f1_threshold_search_"
        + EVALUATION_RUN_NAME
        + ".csv"
    ),
    index=False,
)

membership_metrics.to_csv(
    MEMBERSHIP_METRICS_PATH,
    index=False,
)

test_membership_rows.to_csv(
    MEMBERSHIP_TEST_ROWS_PATH,
    index=False,
)

with open(
    MEMBERSHIP_THRESHOLD_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        {
            "evaluation_run_name":
                EVALUATION_RUN_NAME,
            "score_definition":
                "mean correct-answer log probability",
            "f1_threshold":
                membership_threshold,
            "calibration_recipients_per_group":
                CALIBRATION_RECIPIENTS_PER_GROUP,
            "use_after_unlearning":
                "Reuse this threshold unchanged.",
        },
        handle,
        indent=2,
    )

print("Membership metrics on held-out test recipients:")
display(membership_metrics)

print("Saved fixed threshold:")
print(MEMBERSHIP_THRESHOLD_PATH)

Membership metrics on held-out test recipients:


,evaluation_run_name,score_definition,f1_threshold,calibration_recipients_per_group,test_recipients_per_group,test_questions,f1,precision,recall,balanced_accuracy,auroc,pr_auc
0,all_300_reinforced_v1,mean correct-answer log probability,-1.644165,60,240,6720,0.700429,0.654513,0.753274,0.677827,0.774526,0.803666


Saved fixed threshold:
/content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory/profile_memory_membership_threshold_all_300_reinforced_v1.json


In [51]:
check = focused_qa_examples[
    (focused_qa_examples["recipient_id"] == "V32P-R001887")
    &
    (focused_qa_examples["field"] == "donor_age")
]

display(
    check[
        [
            "recipient_id",
            "field",
            "answer",
            "text",
        ]
    ]
)

,recipient_id,field,answer,text
269,V32P-R001887,donor_age,61,Here is the recipient ID:\nV32P-R001887\n\nWha...


## 19. Final Training and Evaluation Verification

A successful run should have:

- 300 memory-training recipients;
- 300 separate unseen-control recipients;
- 4,200 factual QA examples in the main training phase;
- 1,800 extra focused QA examples across all 300 memory recipients;
- no overlap between memory and control groups;
- no acute-rejection prediction task;
- a saved Qwen profile-memory model and archive;
- saved recipient lists and prompt-contract information;
- saved full memory-versus-unseen evaluation results;
- saved factual-recall tables and membership metrics.

The evaluation results use exact factual recall as the primary measure. F1, precision, recall, balanced accuracy, AUROC, and PR-AUC are secondary membership-memory measures and must not be described as acute-rejection prediction metrics.

In [52]:
final_checks = pd.DataFrame([
    {
        "Check": "300 memory-training recipients",
        "Pass": len(memory_recipient_ids) == 300,
    },
    {
        "Check": "300 reserved unseen controls",
        "Pass": (
            len(final_control_recipient_ids)
            == 300
        ),
    },
    {
        "Check": "Memory and control groups do not overlap",
        "Pass": set(memory_recipient_ids).isdisjoint(
            set(final_control_recipient_ids)
        ),
    },
    {
        "Check": "4,200 main factual QA examples",
        "Pass": len(qa_examples) == 4_200,
    },
    {
        "Check": "300 focused memory recipients",
        "Pass": (
            len(final_memory_recipient_ids)
            == 300
        ),
    },
    {
        "Check": "1,800 focused QA examples",
        "Pass": (
            len(focused_qa_examples)
            == 1_800
        ),
    },
    {
        "Check": "No acute-rejection prediction task",
        "Pass": not qa_examples[
            "question"
        ].str.contains(
            "acute rejection",
            case=False,
            regex=False,
        ).any(),
    },
    {
        "Check": "Final model directory exists",
        "Pass": PROFILE_MODEL_DIR.exists(),
    },
    {
        "Check": "Model archive exists",
        "Pass": MODEL_ARCHIVE.exists(),
    },
    {
        "Check": "Prompt contract saved",
        "Pass": (
            RESULT_DIR
            / "profile_memory_contract.json"
        ).exists(),
    },
    {
        "Check": "Evaluation plan saved",
        "Pass": EVALUATION_PLAN_PATH.exists(),
    },
    {
        "Check": "Full evaluation contains 8,400 rows",
        "Pass": (
            EVALUATION_RESULTS_PATH.exists()
            and len(
                pd.read_csv(
                    EVALUATION_RESULTS_PATH
                )
            ) == 8_400
        ),
    },
    {
        "Check": "Overall recall table saved",
        "Pass": OVERALL_RECALL_PATH.exists(),
    },
    {
        "Check": "Recall-by-field table saved",
        "Pass": FIELD_RECALL_PATH.exists(),
    },
    {
        "Check": "Membership metrics saved",
        "Pass": MEMBERSHIP_METRICS_PATH.exists(),
    },
    {
        "Check": "Fixed F1 threshold saved",
        "Pass": MEMBERSHIP_THRESHOLD_PATH.exists(),
    },
])

display(final_checks)

assert final_checks["Pass"].all()

print(
    "Profile-memory training and evaluation "
    "notebook complete."
)

,Check,Pass
0,300 memory-training recipients,True
1,300 reserved unseen controls,True
2,Memory and control groups do not overlap,True
3,"4,200 main factual QA examples",True
4,300 focused memory recipients,True
5,"1,800 focused QA examples",True
6,No acute-rejection prediction task,True
7,Final model directory exists,True
8,Model archive exists,True
9,Prompt contract saved,True


Profile-memory training and evaluation notebook complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')